<a href="https://colab.research.google.com/github/arunkumar-0512/Ai-Powered-Smart-Email-Classifier-for-Enterprises/blob/main/m1_datapreprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
df = pd.read_csv("email_dataset_full.csv")
df.head()


,id,subject,body,text,category,category_id,split
0,promotions_582,Anniversary Special: Buy one get one free,"As our loyal customer, get exclusive $60 off $...",Anniversary Special: Buy one get one free As o...,promotions,1,train
1,spam_1629,Your Amazon was used on new device,Your $5000 refund is processed. Claim: bit.ly/...,Your Amazon was used on new device Your $5000 ...,spam,3,train
2,spam_322,Re: Your Google inquiry,"Hi, following up about your Google application...","Re: Your Google inquiry Hi, following up about...",spam,3,train
3,social_media_80,Digital Ritual Experience Creation,Cross-cultural ceremony design. Join: virtualr...,Digital Ritual Experience Creation Cross-cultu...,social_media,2,train
4,forum_1351,"Your post was moved to ""Programming Help""","Trending: ""cooking"" (258 comments). View: supp...","Your post was moved to ""Programming Help"" Tren...",forum,0,train


In [2]:
df.columns

Index(['id', 'subject', 'body', 'text', 'category', 'category_id', 'split'], dtype='object')

In [3]:
!pip install nltk

In [4]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [5]:
def remove_signature(text):
    if pd.isna(text):
        return ""
    patterns = [
        r"regards,.*",
        r"thanks,.*",
        r"best regards,.*",
        r"sincerely,.*",
        r"kind regards,.*"
    ]
    for p in patterns:
        text = re.sub(p, "", text, flags=re.IGNORECASE | re.DOTALL)

    return text


In [6]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


In [7]:
stop_words = set(stopwords.words("english"))
def remove_stopwords(text):
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)


In [8]:
def preprocess_email(text):
    text = remove_signature(text)
    text = normalize_text(text)
    text = remove_stopwords(text)
    return text


In [9]:
df["cleaned_email"] = df["text"].apply(preprocess_email)
df[["text", "cleaned_email"]].head()


,text,cleaned_email
0,Anniversary Special: Buy one get one free As o...,anniversary special buy one get one free loyal...
1,Your Amazon was used on new device Your $5000 ...,amazon used new device refund processed claim ...
2,"Re: Your Google inquiry Hi, following up about...",google inquiry hi following google application...
3,Digital Ritual Experience Creation Cross-cultu...,digital ritual experience creation cross cultu...
4,"Your post was moved to ""Programming Help"" Tren...",post moved programming help trending cooking c...


In [10]:
def label_urgency(text):
    if any(word in text for word in ["urgent", "asap", "immediately", "critical"]):
        return "High"
    elif any(word in text for word in ["soon", "priority"]):
        return "Medium"
    else:
        return "Low"

df["urgency"] = df["cleaned_email"].apply(label_urgency)
df[["cleaned_email", "category", "urgency"]].head()


,cleaned_email,category,urgency
0,anniversary special buy one get one free loyal...,promotions,Low
1,amazon used new device refund processed claim ...,spam,Low
2,google inquiry hi following google application...,spam,Low
3,digital ritual experience creation cross cultu...,social_media,Low
4,post moved programming help trending cooking c...,forum,Low


In [11]:
df.to_csv("email_dataset_preprocessed.csv", index=False)


Final dataset shape: (13477, 9)
